### **Student Information**
GitHub ID: jingwenliu0113-afk

Kaggle name: JineWen LIU

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

Before writing any cleaning rules, I first inspected the raw Kaggle competition files in `data/kaggle/` (`final_posts.json`, `data_identification.csv`, and `emotion.csv`) joined on `post_id`:

* **Dataset Shape & Class Distribution (Section 1.2):** The dataset contains **3,360 training posts** and **1,440 test posts** across **six emotion classes**. The training labels are noticeably imbalanced: `joy` (875), `anger` (674), `sadness` (628), `disgust` (454), `surprise` (441), and `fear` (288 — less than one-third of `joy`), which makes **Macro F1** much stricter than plain accuracy on the smaller classes.
* **Exploratory Data Checks (Section 1.3):**
  * **Length distribution:** Posts have a median of **13 words** (95th percentile **33 words**, max **508 words**).
  * **Duplicates & label noise:** There are **100 duplicated text rows** inside the training set, including **16 texts that carry conflicting emotion labels**, and **59 test texts** that also appear in the training split.
  * **Mixed data sources:** Posts come from multiple platforms—including Reddit comments with masked names `[NAME]` (**7.0%** of training posts), tweets with masked handles `@<username>` (**3.0%** of training posts), news headlines, and dialogue lines.
* **Model-Specific Cleaning Variants & Shared Folds (Section 1.4):** Because pre-trained Transformers and bag-of-n-gram linear models read text differently, I created two separate cleaned columns alongside a single shared 5-fold stratified split (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`):
  * `clean_for_transformer` (`text_tr`): Keeps casing, punctuation, and emojis intact because Transformer tokenizers extract strong emotional cues from capitalization, exclamation marks, and emojis. It only decodes HTML entities (`html.unescape`, e.g., `&amp;` $\rightarrow$ `&`), replaces `@<username>` with `@user`, and collapses extra whitespace.
  * `clean_for_tfidf` (`text_tfidf`): Applies `clean_for_transformer` first, then collapses any remaining `@\w+` handle into `@user` and shortens 4 or more repeated identical characters down to 3 (`(.)\1{3,}` $\rightarrow$ `\1\1\1`) so stretched spellings like `"soooooo"` and `"sooo"` map into the same feature column.

---

### 1.2 Feature Engineering Steps

I engineered two complementary feature families—sparse surface n-grams for linear classifiers and sub-word token sequences for fine-tuned Transformers:

* **Sparse Word + Character N-Gram TF-IDF (`build_tfidf()`, Section 2.1):** Using `make_union`, I concatenated two `TfidfVectorizer` representations into a combined **`(3360, 65300)`** sparse matrix when fitted on all training posts:
  1. **Word 1–2 grams:** `ngram_range=(1, 2)`, `sublinear_tf=True`, and custom `token_pattern=r"(?u)\b\w+\b|[!?]"` so standalone `!` and `?` punctuation marks are kept as features rather than stripped.
  2. **Character 2–5 grams inside word boundaries:** `analyzer="char_wb"`, `ngram_range=(2, 5)`, `min_df=2`, `sublinear_tf=True`. Character n-grams pick up morphological roots and suffixes (such as `"hate"` inside `"hateful"`), typos, and elongated words that whole-word columns miss.
* **Dense Contextual Sub-Word Tokens (Section 2.2):** For the Transformer models, feature engineering consists of choosing the pre-trained checkpoints and sequence length. Tokenizing the training set with `FacebookAI/roberta-base` showed a median of **18 sub-word tokens** per post and a 99th percentile of **111 tokens**, with only **27 of 3,360 posts** exceeding **128 tokens**. Setting `MAX_LEN = 128` therefore covers over **99.2%** of the dataset while keeping GPU/MPS memory and training time manageable.

---

### 1.3 Explanation of Your Model

All models were trained and evaluated on the **exact same 5 stratified folds** using **out-of-fold (OOF) Macro F1**, allowing direct, leak-free comparisons and probability blending:

| Stage / Model | Representation / Settings | 5-Fold OOF Macro F1 |
| --- | --- | ---: |
| **Linear Baseline 1** | Word 1–2g TF-IDF + `LinearSVC(C=0.3, class_weight="balanced")` | `0.4826` |
| **Linear Baseline 2** | Word 1–2g + Char 2–5g TF-IDF + `LinearSVC(C=0.1, class_weight="balanced")` | `0.5125` |
| **Linear Probabilities (`oof_tfidf`)** | Word 1–2g + Char 2–5g TF-IDF + `LogisticRegression(C=5, class_weight="balanced")` | `0.5010` |
| **Transformer 1 (weight = 1)** | `FacebookAI/roberta-base` (`lr = 2e-5`, 4 epochs, batch size 16) | `0.5964` |
| **Transformer 2 (weight = 2)** | `FacebookAI/roberta-large` (`lr = 1e-5`, 4 epochs, batch size 16) | `0.6111` |
| **Transformer 3 (weight = 2)** | `cardiffnlp/twitter-roberta-large-2022-154m` (`lr = 1e-5`, 4 epochs, batch size 16) | `0.6180` |
| **3-Transformer Weighted Blend** | `1 * roberta-base + 2 * roberta-large + 2 * twitter-roberta-large` | `0.6171` |
| **Final Hybrid Blend (Selected)** | **`0.7 * Transformer Blend + 0.3 * TF-IDF LogReg` (`USE_BIAS = False`)** | **`0.6251`** |

1. **TF-IDF + Linear Baselines (Section 3.1):** Adding character 2–5 grams (`char_wb`) to word 1–2 grams raised 5-fold OOF Macro F1 on `LinearSVC` from **`0.4826`** to **`0.5125`** (+0.0299), crossing the 0.50 competition baseline with a linear model alone. Fitting `LogisticRegression(C=5, class_weight="balanced", max_iter=3000)` on the same hybrid TF-IDF features achieved **`0.5010`** OOF Macro F1 and produced calibrated 6-class probability predictions (`oof_tfidf` and `test_tfidf`) for blending.
2. **5-Fold Transformer Fine-Tuning (Section 3.2):** I fine-tuned three general pre-trained checkpoints on Apple Silicon (`device: mps`) for `EPOCHS = 4` (`BATCH_SIZE = 16`) with `AdamW` (weight decay `0.01` excluding biases and `LayerNorm.weight`), 10% linear warm-up followed by linear decay, gradient norm clipping at `1.0`, and explicit `dtype=torch.float32`. After each epoch, the loop scored validation Macro F1 and kept the best epoch's validation and test probabilities per fold. Scaling from `roberta-base` (**`0.5964`**) to `roberta-large` (**`0.6111`**) and domain-matched `cardiffnlp/twitter-roberta-large-2022-154m` (**`0.6180`**, pre-trained on 154M tweets) steadily improved OOF Macro F1.
3. **Probability Blending & Honest Nested-CV Bias Verification (Sections 3.3 & 3.4):**
   * Combining the three Transformers with weights `1 : 2 : 2` gave **`0.6171`** OOF Macro F1. Mixing in the `word+char + LogReg` probabilities (`(1 - w) * oof_tr + w * oof_tfidf`) improved OOF Macro F1 across every weight tested (`w=0.1` $\rightarrow$ `0.6220`, `w=0.2` $\rightarrow$ `0.6242`, **`w=0.3` $\rightarrow$ `0.6251`**, `w=0.4` $\rightarrow$ `0.6233`, `w=0.5` $\rightarrow$ `0.6216`), demonstrating that sparse surface n-grams and dense contextual Transformers make complementary errors.
   * Because `fear` (288 rows) and `disgust` (454 rows) are smaller classes, I also tested coordinate-ascent per-class log-probability bias tuning (`tune_bias`). To check whether tuning 6 class offsets actually generalizes rather than overfitting OOF predictions, I evaluated it with **nested CV** (tuning biases on 4 folds' OOF probabilities and scoring on the held-out 5th fold). Nested CV dropped Macro F1 from **`0.6251`** to **`0.6195`**, so the code automatically set `USE_BIAS = False` and kept unmodified probabilities (`0.6251` OOF Macro F1, `0.649` accuracy; per-class F1: `joy 0.777`, `sadness 0.657`, `fear 0.638`, `surprise 0.625`, `anger 0.617`, `disgust 0.436`).
   * Finally, Section 3.4 wrote `data/kaggle/submission.csv` (`1,440` rows: `joy 388`, `anger 363`, `sadness 277`, `surprise 174`, `disgust 129`, `fear 109`) with columns `id,label` verified against `sampleSubmission.csv`.

---

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

During the competition, I worked with AI to build the pipeline, run experiments on my Mac (`mps`), and submit versions (`v1` to `v5`) to Kaggle with my approval. The prompts below are **paraphrased (改寫自原指令)** from my actual instructions during the session:

* **Prompt 1 — *(Paraphrased / 改寫自原指令)* (Initial baseline and first Kaggle submission — `v1`):**
  > *"幫我完成 Lab 2 的 Homework 競賽部分，讀取 `data/kaggle/` 的資料建立模型並產生預測檔上傳到 Kaggle。"*
  * **What AI suggested & did:** Built a 5-fold `StratifiedKFold` pipeline, compared `word 1-2g + LinearSVC` (`0.4826` OOF Macro F1) against adding `char_wb` 2–5 grams (`0.5125` OOF Macro F1), fine-tuned `FacebookAI/roberta-base` (`0.5964` OOF Macro F1), and blended `roberta-base` with `LogisticRegression` on the word+char TF-IDF features (with TF-IDF weight `0.2`). When submitting to Kaggle (after I approved uploading), the first upload was rejected by Kaggle with `"ID column id not found in submission"` because `sampleSubmission.csv` used `post_id` as its first column header; AI spotted the mismatch from the error message, changed the output header to `id,label` on its own (`submission_v1_roberta-base_id.csv`), and completed the upload.
  * **My decision:** I approved submitting `v1` (`roberta-base` + `0.2` TF-IDF) to verify our pipeline on the live leaderboard and secure an initial score comfortably above the 0.50 baseline.

* **Prompt 2 — *(Paraphrased / 改寫自原指令)* (Scaling up to larger Transformers — `v2` & `v3`):**
  > *"繼續幫我把分數往上推，試試看加入更大的預訓練模型做 ensemble。"*
  * **What AI suggested & did:** While testing larger checkpoints (`DeBERTa-v3`, `roberta-large`), AI found that `transformers 5` keeps checkpoints in their stored `float16` precision by default, causing AdamW's `eps` to underflow into `NaN` during training; AI fixed this numerical issue by loading models in `dtype=torch.float32` (this only makes training run normally without `NaN`, whereas the actual score improvements come from the stronger models below). To raise the score, AI trained two additional 5-fold models:
    * **`v2` (`submission_v2_large-base.csv`):** Combined `FacebookAI/roberta-large` (`0.6111` OOF Macro F1) with `roberta-base`.
    * **`v3` (`submission_v3_large-twitter-base.csv`):** Added domain-pre-trained `cardiffnlp/twitter-roberta-large-2022-154m` (`0.6180` OOF Macro F1) to form a pure 3-Transformer weighted ensemble (`roberta-large ×2`, `twitter-roberta-large-2022-154m ×2`, `roberta-base ×1`, with **no TF-IDF**).
  * **My decision:** Based on the steady OOF Macro F1 gains (`0.5964` $\rightarrow$ `0.6111` $\rightarrow$ `0.6180`), I approved uploading `v2` and `v3` to track how the pure Transformer ensembles performed on the leaderboard.

* **Prompt 3 — *(Paraphrased / 改寫自原指令)* (Consolidating the hybrid notebook pipeline, testing bias tuning, and final selection — `v4` & `v5`):**
  > *"幫我把完整的流程整理進 `DM2026-Lab2-Homework.ipynb` 裡實際跑完（包含混入 TF-IDF 跟測試 per-class bias 調整），另外也試試看能不能疊更多模型（v5），最後跟我討論要選哪兩個版本當最終提交。"*
  * **What AI suggested & did:**
    1. Ran the complete notebook pipeline (`v4`, `submission_v4_notebook.csv` / `submission.csv`), which takes the 3-Transformer weighted blend (`1 : 2 : 2`, OOF Macro F1 `0.6171`) and mixes in the `word+char + LogReg` TF-IDF probabilities at weight **`W_TFIDF = 0.3`**, raising OOF Macro F1 to **`0.6251`**.
    2. Tested coordinate-ascent per-class log-probability bias tuning (`tune_bias`) using nested 5-fold CV (fitting biases on 4 folds, scoring on the 5th). Nested CV showed that bias tuning lowered OOF Macro F1 from `0.6251` to `0.6195`.
    3. Also tested a larger multi-checkpoint + TF-IDF blend (`v5`, `submission_v5_6large-tfidf.csv`).
  * **My decision:**
    1. **Kept `USE_BIAS = False`:** Since nested CV showed that tuning 6 class offsets overfits the OOF predictions (`0.6195 < 0.6251`), we did not apply per-class bias offsets.
    2. **Decided whether to upload each version and chose `v3 + v4` for final scoring:** After reviewing all versions (`v1` through `v5`), I decided on **`v3` (`submission_v3_large-twitter-base.csv`) and `v4` (`submission_v4_notebook.csv` / `submission.csv`)** as the two final submissions—`v3` is the pure 3-Transformer ensemble (`roberta-large ×2`, `twitter-roberta-large ×2`, `roberta-base ×1`, without TF-IDF), while `v4` adds the `0.3`-weighted word+char TF-IDF LogReg blend (`0.6251` OOF Macro F1) and matches this notebook's exact executed output.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### 1.1 Setup: imports, paths, device and random seeds
import html
import math
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

transformers.logging.set_verbosity_error()   # hide the "newly initialized classifier" load report
transformers.utils.logging.disable_progress_bar()

DATA_DIR = Path("data/kaggle")               # unzipped competition files (kept out of git)
SUBMISSION_PATH = DATA_DIR / "submission.csv"
SEED = 42
N_FOLDS = 5

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: mps


In [2]:
### 1.2 Load the competition files and join them on post_id
posts = pd.read_json(DATA_DIR / "final_posts.json")             # post_id, text  (train + test)
split = pd.read_csv(DATA_DIR / "data_identification.csv")       # post_id, set   ("train" / "test")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")                 # post_id, label (train only)

df = posts.merge(split, on="post_id").merge(emotion, on="post_id", how="left")
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)

LABELS = sorted(train_df["label"].unique())                     # one fixed class order for every model
label_to_id = {label: i for i, label in enumerate(LABELS)}
y = train_df["label"].map(label_to_id).to_numpy()

print(f"train {train_df.shape}, test {test_df.shape}")
print(train_df["label"].value_counts().to_string())

train (3360, 4), test (1440, 4)
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288


In [3]:
### 1.3 Data checks that shape the preprocessing choices
n_words = train_df["text"].str.split().str.len()
print("words per post: median", int(n_words.median()), "| 95th percentile", int(n_words.quantile(0.95)), "| max", n_words.max())

duplicated = train_df["text"].duplicated(keep=False)
conflicting = train_df[duplicated].groupby("text")["label"].nunique().gt(1).sum()
print(f"duplicated texts in train: {duplicated.sum()} rows; {conflicting} texts carry conflicting labels")
print("test texts that also occur in train:", test_df["text"].isin(set(train_df["text"])).sum())

# The posts come from several sources: Reddit comments with names masked as [NAME],
# tweets with handles masked as @<username>, headlines and dialogue lines.
name_share = train_df["text"].str.contains("[NAME]", regex=False).mean()
handle_share = train_df["text"].str.contains("@<username>", regex=False).mean()
print(f"posts with [NAME]: {name_share:.1%} | posts with @<username>: {handle_share:.1%}")

words per post: median 13 | 95th percentile 33 | max 508
duplicated texts in train: 100 rows; 16 texts carry conflicting labels
test texts that also occur in train: 59
posts with [NAME]: 7.0% | posts with @<username>: 3.0%


In [4]:
### 1.4 Cleaning, one variant per model family, and the shared CV folds
# The transformer gets almost raw text: case, punctuation and emoji carry emotion it can read.
# TF-IDF gets the same text with all user handles merged and stretched letters shortened,
# so "soooooo" and "sooo" fall into the same column.
def clean_for_transformer(text: str) -> str:
    text = html.unescape(text)                      # &amp; -> &, &lt; -> <
    text = re.sub(r"@<username>", "@user", text)    # anonymised Twitter handles
    return re.sub(r"\s+", " ", text).strip()


def clean_for_tfidf(text: str) -> str:
    text = clean_for_transformer(text)
    text = re.sub(r"@\w+", " @user ", text)         # any remaining handle -> one token
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)    # 4+ repeated characters -> 3
    return re.sub(r"\s+", " ", text).strip()


for frame in (train_df, test_df):
    frame["text_tr"] = frame["text"].map(clean_for_transformer)
    frame["text_tfidf"] = frame["text"].map(clean_for_tfidf)
print(train_df.loc[train_df["text"] != train_df["text_tfidf"], ["text", "text_tfidf"]].head(3).to_string(max_colwidth=60))

# One set of stratified folds shared by every model, so their out-of-fold predictions line up.
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))

                                                           text                                                   text_tfidf
35  Dude yeah. Worst part, guy comes in the morning. So alre...  Dude yeah. Worst part, guy comes in the morning. So alre...
38  I’m sure losing obj will help [NAME] get out of his slum...  I’m sure losing obj will help [NAME] get out of his slum...
53  Terrazene one ken bessede snow fire someone drink salt y...  Terrazene one ken bessede snow fire someone drink salt y...


## 2. Feature Engineering Steps

In [5]:
### 2.1 Sparse features: TF-IDF on word 1-2 grams + character 2-5 grams
# Character n-grams inside word boundaries pick up misspellings, elongations and word pieces
# ("hate" inside "hateful") that whole-word columns miss; section 3.1 compares the two.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union


def build_tfidf():
    return make_union(
        TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b|[!?]"),  # keep ! and ?
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    )


print("TF-IDF matrix when fitted on all training posts:", build_tfidf().fit_transform(train_df["text_tfidf"]).shape)

TF-IDF matrix when fitted on all training posts: (3360, 65300)


In [6]:
### 2.2 Dense contextual features: transformer sub-word tokens
# A fine-tuned transformer learns its own features from sub-word tokens, so feature engineering
# here is the choice of checkpoint and sequence length. All checkpoints below are general
# pre-trained English models (no emotion labels seen during pre-training).
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TRANSFORMER_MODELS = [("FacebookAI/roberta-base", 2e-5, 1), ("FacebookAI/roberta-large", 1e-5, 2), ("cardiffnlp/twitter-roberta-large-2022-154m", 1e-5, 2)]   # (checkpoint, learning rate, blend weight)
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODELS[0][0])
lengths = pd.Series([len(ids) for ids in tokenizer(train_df["text_tr"].tolist())["input_ids"]])
print("tokens per post: median", int(lengths.median()), "| 99th percentile", int(lengths.quantile(0.99)),
      f"| posts longer than MAX_LEN={MAX_LEN}:", int((lengths > MAX_LEN).sum()))

tokens per post: median 18 | 99th percentile 111 | posts longer than MAX_LEN=128: 27


## 3. Model Implementation Steps

In [7]:
### 3.1 Baselines: TF-IDF + linear models (5-fold CV, macro F1)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

texts = train_df["text_tfidf"]
word_only_svm = lambda: make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b|[!?]"),
    LinearSVC(C=0.3, class_weight="balanced"))
word_char_svm = lambda: make_pipeline(build_tfidf(), LinearSVC(C=0.1, class_weight="balanced"))
for name, make_model in [("word 1-2g + LinearSVC", word_only_svm), ("word+char + LinearSVC", word_char_svm)]:
    oof = np.zeros(len(y), dtype=int)
    for train_idx, val_idx in folds:
        oof[val_idx] = make_model().fit(texts.iloc[train_idx], y[train_idx]).predict(texts.iloc[val_idx])
    print(f"{name:24s} OOF macro F1 = {f1_score(y, oof, average='macro'):.4f}")

# Logistic regression on the same features gives class probabilities, which the final blend needs.
oof_tfidf = np.zeros((len(y), len(LABELS)))
test_tfidf = np.zeros((len(test_df), len(LABELS)))
for train_idx, val_idx in folds:
    model = make_pipeline(build_tfidf(), LogisticRegression(C=5, class_weight="balanced", max_iter=3000))
    model.fit(texts.iloc[train_idx], y[train_idx])
    oof_tfidf[val_idx] = model.predict_proba(texts.iloc[val_idx])
    test_tfidf += model.predict_proba(test_df["text_tfidf"]) / N_FOLDS
print(f"{'word+char + LogReg':24s} OOF macro F1 = {f1_score(y, oof_tfidf.argmax(1), average='macro'):.4f}")

word 1-2g + LinearSVC    OOF macro F1 = 0.4826


word+char + LinearSVC    OOF macro F1 = 0.5125


word+char + LogReg       OOF macro F1 = 0.5010


In [8]:
### 3.2 Fine-tuning the transformer(s) on the same 5 folds
# Per fold: start from the pre-trained weights, train on 4/5 of the labelled posts (AdamW,
# linear warm-up then linear decay), score macro F1 on the held-out 1/5 after every epoch, and
# keep the best epoch's validation and test probabilities. The fold models' test probabilities
# are averaged; their validation probabilities form the out-of-fold (OOF) estimate.
BATCH_SIZE, EPOCHS = 16, 4


def batches(enc, indices, rng=None):
    """Yield (row indices, padded tensors on DEVICE); shuffled when an rng is given."""
    indices = np.array(indices)
    if rng is not None:
        rng.shuffle(indices)
    for start in range(0, len(indices), BATCH_SIZE):
        rows = indices[start:start + BATCH_SIZE]
        features = [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i]} for i in rows]
        padded = tokenizer.pad(features, return_tensors="pt")   # pad to the longest post in the batch
        yield rows, {k: v.to(DEVICE) for k, v in padded.items()}


@torch.no_grad()
def predict_proba(model, enc, indices):
    model.eval()
    return np.concatenate([torch.softmax(model(**inputs).logits.float(), dim=-1).cpu().numpy()
                           for _, inputs in batches(enc, indices)])


def train_fold(checkpoint, lr, fold, train_idx, val_idx, train_enc, test_enc):
    torch.manual_seed(SEED + fold)
    rng = np.random.default_rng(SEED + fold)
    # float32 on purpose: transformers 5 keeps a checkpoint's stored precision, and a float16
    # checkpoint (DeBERTa-v3) makes AdamW's eps underflow so every weight becomes NaN.
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(LABELS), dtype=torch.float32).to(DEVICE)

    no_decay = ("bias", "LayerNorm.weight")      # standard: no weight decay on biases and LayerNorm
    optimizer = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": 0.01},
        {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0},
    ], lr=lr)
    total_steps = EPOCHS * math.ceil(len(train_idx) / BATCH_SIZE)
    warmup = int(0.1 * total_steps)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: min((s + 1) / max(1, warmup), max(0.0, (total_steps - s) / max(1, total_steps - warmup))))

    best_f1, best_val, best_test = -1.0, None, None
    for epoch in range(EPOCHS):
        model.train()
        start, loss_sum = time.time(), 0.0
        for rows, inputs in batches(train_enc, train_idx, rng):
            loss = torch.nn.functional.cross_entropy(model(**inputs).logits, torch.tensor(y[rows], device=DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            loss_sum += loss.item() * len(rows)
        val_probs = predict_proba(model, train_enc, val_idx)
        f1 = f1_score(y[val_idx], val_probs.argmax(1), average="macro")
        print(f"  fold {fold} epoch {epoch + 1}/{EPOCHS}: train loss {loss_sum / len(train_idx):.4f}, "
              f"val macro F1 {f1:.4f} ({time.time() - start:.0f}s)")
        if f1 > best_f1:
            best_f1, best_val = f1, val_probs
            best_test = predict_proba(model, test_enc, range(len(test_df)))
    del model, optimizer
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    return best_f1, best_val, best_test


oof_transformer, test_transformer = {}, {}
for checkpoint, lr, _ in TRANSFORMER_MODELS:
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    train_enc = tokenizer(train_df["text_tr"].tolist(), truncation=True, max_length=MAX_LEN)
    test_enc = tokenizer(test_df["text_tr"].tolist(), truncation=True, max_length=MAX_LEN)
    oof, test_probs = np.zeros((len(y), len(LABELS))), np.zeros((len(test_df), len(LABELS)))
    print(f"=== {checkpoint} (lr {lr})")
    for fold, (train_idx, val_idx) in enumerate(folds):
        best_f1, val_probs, fold_test = train_fold(checkpoint, lr, fold, train_idx, val_idx, train_enc, test_enc)
        oof[val_idx] = val_probs
        test_probs += fold_test / N_FOLDS
        print(f"  fold {fold}: best val macro F1 {best_f1:.4f}")
    oof_transformer[checkpoint], test_transformer[checkpoint] = oof, test_probs
    print(f"{checkpoint} OOF macro F1 = {f1_score(y, oof.argmax(1), average='macro'):.4f}\n")

=== FacebookAI/roberta-base (lr 2e-05)


  fold 0 epoch 1/4: train loss 1.5592, val macro F1 0.5177 (64s)


  fold 0 epoch 2/4: train loss 1.0476, val macro F1 0.5757 (62s)


  fold 0 epoch 3/4: train loss 0.7893, val macro F1 0.5723 (63s)


  fold 0 epoch 4/4: train loss 0.6314, val macro F1 0.5808 (63s)


  fold 0: best val macro F1 0.5808


... (65 outputs left out)

  fold 4 epoch 2/4: train loss 1.1314, val macro F1 0.5690 (180s)


  fold 4 epoch 3/4: train loss 0.8164, val macro F1 0.6044 (181s)


  fold 4 epoch 4/4: train loss 0.6274, val macro F1 0.5909 (181s)


  fold 4: best val macro F1 0.6044
cardiffnlp/twitter-roberta-large-2022-154m OOF macro F1 = 0.6180



In [9]:
### 3.3 Blending the models and tuning per-class biases for macro F1
# (a) Weighted average of the transformers (the two large models count double: each scored
#     higher than roberta-base in CV), then mix in TF-IDF with a weight chosen on the OOF predictions.
weights = {checkpoint: w for checkpoint, _, w in TRANSFORMER_MODELS}
total = sum(weights.values())
oof_tr = sum(weights[c] * p for c, p in oof_transformer.items()) / total
test_tr = sum(weights[c] * p for c, p in test_transformer.items()) / total
print(f"transformer blend {weights}: OOF macro F1 {f1_score(y, oof_tr.argmax(1), average='macro'):.4f}")
scores = {w: f1_score(y, ((1 - w) * oof_tr + w * oof_tfidf).argmax(1), average="macro") for w in np.arange(0, 0.55, 0.1)}
for w, s in scores.items():
    print(f"TF-IDF weight {w:.1f}: OOF macro F1 {s:.4f}")
W_TFIDF = max(scores, key=scores.get)
oof_blend = (1 - W_TFIDF) * oof_tr + W_TFIDF * oof_tfidf


# (b) Macro F1 weighs every class equally, so a small additive bias per class on the
#     log-probabilities can buy recall on rare classes (fear has 288 posts) for a little
#     accuracy on frequent ones.
def tune_bias(probs, target, rounds=3, grid=np.linspace(-1.0, 1.0, 21)):
    """Coordinate ascent on per-class log-probability biases, maximising macro F1."""
    log_p, bias = np.log(probs + 1e-9), np.zeros(probs.shape[1])
    for _ in range(rounds):
        for c in range(probs.shape[1]):
            trial_scores = []
            for g in grid:
                trial = bias.copy()
                trial[c] = g
                trial_scores.append(f1_score(target, (log_p + trial).argmax(1), average="macro"))
            bias[c] = grid[int(np.argmax(trial_scores))]
    return bias


# Honest estimate of what the biases add: tune on four folds' OOF rows, score on the fifth.
nested_pred = np.empty(len(y), dtype=int)
for train_idx, val_idx in folds:
    nested_pred[val_idx] = (np.log(oof_blend[val_idx] + 1e-9) + tune_bias(oof_blend[train_idx], y[train_idx])).argmax(1)
plain_f1 = f1_score(y, oof_blend.argmax(1), average="macro")
nested_f1 = f1_score(y, nested_pred, average="macro")
print(f"\nblend (TF-IDF weight {W_TFIDF:.1f}), no bias : OOF macro F1 {plain_f1:.4f}")
print(f"blend + per-class bias (nested CV) : OOF macro F1 {nested_f1:.4f}")

USE_BIAS = nested_f1 > plain_f1
BIAS = tune_bias(oof_blend, y) if USE_BIAS else np.zeros(len(LABELS))
print("use bias:", USE_BIAS, "|", dict(zip(LABELS, BIAS.round(2))))

final_oof = nested_pred if USE_BIAS else oof_blend.argmax(1)
print("\n" + classification_report(y, final_oof, target_names=LABELS, digits=3))
print(pd.DataFrame(confusion_matrix(y, final_oof), index=[f"true {c}" for c in LABELS], columns=[f"pred {c}" for c in LABELS]))

transformer blend {'FacebookAI/roberta-base': 1, 'FacebookAI/roberta-large': 2, 'cardiffnlp/twitter-roberta-large-2022-154m': 2}: OOF macro F1 0.6171
TF-IDF weight 0.0: OOF macro F1 0.6171
TF-IDF weight 0.1: OOF macro F1 0.6220
TF-IDF weight 0.2: OOF macro F1 0.6242
TF-IDF weight 0.3: OOF macro F1 0.6251
TF-IDF weight 0.4: OOF macro F1 0.6233
TF-IDF weight 0.5: OOF macro F1 0.6216



blend (TF-IDF weight 0.3), no bias : OOF macro F1 0.6251
blend + per-class bias (nested CV) : OOF macro F1 0.6195
use bias: False | {'anger': np.float64(0.0), 'disgust': np.float64(0.0), 'fear': np.float64(0.0), 'joy': np.float64(0.0), 'sadness': np.float64(0.0), 'surprise': np.float64(0.0)}

              precision    recall  f1-score   support

       anger      0.571     0.672     0.617       674
     disgust      0.521     0.374     0.436       454
        fear      0.716     0.576     0.638       288
         joy      0.762     0.792     0.777       875
     sadness      0.620     0.699     0.657       628
    surprise      0.665     0.590     0.625       441

    accuracy                          0.649      3360
   macro avg      0.642     0.617     0.625      3360
weighted avg      0.648     0.649     0.645      3360

               pred anger  pred disgust  pred fear  pred joy  pred sadness  \
true anger            453            59         20        41            71   
true d

In [10]:
### 3.4 Predict the test set and write the submission file
final_test = (1 - W_TFIDF) * test_tr + W_TFIDF * test_tfidf
test_pred = (np.log(final_test + 1e-9) + BIAS).argmax(1)
# Kaggle's scorer expects the columns "id,label" (as on the Overview page). sampleSubmission.csv
# names the first column "post_id", but a file with that header is rejected with
# "ID column id not found in submission".
submission = pd.DataFrame({"id": test_df["post_id"], "label": [LABELS[i] for i in test_pred]})

sample = pd.read_csv(DATA_DIR / "sampleSubmission.csv")         # same 1,440 test ids as the template
assert len(submission) == len(sample) and set(submission["id"]) == set(sample["post_id"])
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"saved {SUBMISSION_PATH} ({len(submission)} rows)")
print(submission["label"].value_counts().to_string())

saved data/kaggle/submission.csv (1440 rows)
label
joy         388
anger       363
sadness     277
surprise    174
disgust     129
fear        109
